# API v2 - Azure Endpoint Predictions
This notebook sends data to the deployed Azure endpoint and returns model predictions.

In [2]:
import json
import pandas as pd
import requests

REQUIRED_COLUMNS = [
    "StandardCost",
    "ListPrice",
    "Weight",
    "ProductCategoryID",
    "ProductModelID",
]

data = pd.read_csv("prediction_instances.csv")

missing = [c for c in REQUIRED_COLUMNS if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns in input CSV: {missing}")

data = data[REQUIRED_COLUMNS].copy()
data.head()

,StandardCost,ListPrice,Weight,ProductCategoryID,ProductModelID
0,12.50,24.99,0.45,1,101
1,18.20,34.50,0.62,1,102
2,25.00,49.99,0.80,2,103
3,31.75,59.95,1.05,2,104
4,14.10,27.80,0.50,3,105


In [3]:
with open("uri.json", "r") as f:
    scoring_uri = json.load(f)["URI"][0]

payload = {
    "data": [data.to_dict(orient="list")]
}

headers = {"Content-Type": "application/json"}
response = requests.post(
    scoring_uri,
    headers=headers,
    data=json.dumps(payload),
    timeout=60,
)

print("Status code:", response.status_code)
print("Raw response:", response.text)

Status code: 200
Raw response: "[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


In [ ]:
if response.status_code != 200:
    raise RuntimeError(f"Request failed ({response.status_code}): {response.text}")

body = response.json()

while isinstance(body, str):
    body = json.loads(body)

if isinstance(body, dict):
    if "error" in body:
        raise RuntimeError(f"Endpoint error: {body['error']}")
    if "predictions" in body:
        predictions = body["predictions"]
    else:
        raise RuntimeError(f"Unexpected response shape: {body}")

elif isinstance(body, list):
    predictions = body

else:
    raise RuntimeError(f"Unexpected response type: {type(body).__name__}")

if len(predictions) != len(data):
    raise RuntimeError(
        f"Prediction count mismatch. Got {len(predictions)} for {len(data)} rows."
)

result = data.copy()
result["Prediction"] = predictions
result

RuntimeError: Unexpected response type: str